In [ ]:
import pandas as pd
import numpy as np
import os
from PIL import Image

def np_CountUpContinuingOnes(b_arr):
    left = np.arange(len(b_arr))
    left[b_arr > 0] = 0
    left = np.maximum.accumulate(left)

    rev_arr = b_arr[::-1]
    right = np.arange(len(rev_arr))
    right[rev_arr > 0] = 0
    right = np.maximum.accumulate(right)
    right = len(rev_arr) - 1 - right[::-1]

    return right - left - 1

def ExtractBreast(img):
    img_copy = img.copy()
    img = np.where(img <= 40, 0, img)
    height, _ = img.shape

    y_a = height // 2 + int(height * 0.4)
    y_b = height // 2 - int(height * 0.4)
    b_arr = img[y_b:y_a].std(axis=0) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    col_ind = np.where(continuing_ones == continuing_ones.max())[0]
    img = img[:, col_ind]

    _, width = img.shape
    x_a = width // 2 + int(width * 0.4)
    x_b = width // 2 - int(width * 0.4)
    b_arr = img[:, x_b:x_a].std(axis=1) != 0
    continuing_ones = np_CountUpContinuingOnes(b_arr)
    row_ind = np.where(continuing_ones == continuing_ones.max())[0]

    return img_copy[row_ind][:, col_ind]

# 定义路径
train_csv_file = '/Volumes/CSAW-M/labels/CSAW-M_train.csv'
test_csv_file = '/Volumes/CSAW-M/labels/CSAW-M_test.csv'
image_dir = '/Volumes/CSAW-M/images/preprocessed'
output_base_dir = '../classification_data/CSAW-M'
split_csv_path = '../classification_data/classification_split.csv'

# 读取数据划分CSV文件
split_df = pd.read_csv(split_csv_path)
# 只保留CSAW-M数据集
split_df = split_df[split_df['dataset'] == 'CSAW-M']

# 读取并合并两个 CSV 文件
train_df = pd.read_csv(train_csv_file, delimiter=';')
test_df = pd.read_csv(test_csv_file, delimiter=';')
combined_df = pd.concat([train_df, test_df], ignore_index=True)

# 合并数据划分信息
combined_df = pd.merge(combined_df, split_df[['data_name', 'data_split']], 
                      left_on=combined_df['Filename'].apply(lambda x: x.split('.')[0]), 
                      right_on='data_name', how='inner')

def process_and_save():
    for index, row in combined_df.iterrows():
        filename = row['Filename']
        data_name = filename.split('.')[0]
        data_split = row['data_split']
        label = str(row['Label']).replace(' ', '')
        
        # 构建图像路径
        image_path = os.path.join(image_dir, filename.split('_')[0], filename)
        
        # 打开图像并处理
        try:
            image = Image.open(image_path)
            image_array = np.array(image)
            image = ExtractBreast(image_array)
            
            # 保存预处理后的图像到新的文件夹
            output_dir = os.path.join(output_base_dir, data_split, data_name)
            os.makedirs(output_dir, exist_ok=True)
            output_image_path = os.path.join(output_dir, 'img.jpg')
            Image.fromarray(image).save(output_image_path)
            
            # 保存Label和Cancer到info_dict.npy
            info_dict = {
                'Masking': label
            }
            np.save(os.path.join(output_dir, 'info_dict.npy'), info_dict)
            print(f'{output_image_path} has been saved')
        except FileNotFoundError:
            print(f"Image not found: {image_path}")
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")

# 处理并保存数据
process_and_save()
print("Processing complete.")